Feature selection using Mutual info Classification

Note: always make sure that your model is being trained by features that impacts the target feature.
linear relatioships are not the only important consideration.

if feature mutal score for a column is 0.0, it means that feature doesnt impact the target feature. 

In [1]:
# Loading our datas
import pandas as pd
import numpy as np

data_files = {
    'sessions': ('session_time',),
    'riders': ('signup_date',),
    'trips': ('pickup_time', 'dropoff_time'),
    'promotions': ('start_date', 'end_date'),
    'drivers': ('signup_date', 'last_active')
}

dfs={}
for table, date_cols in data_files.items():
    dfs[table] = pd.read_csv(f'../data/{table}.csv', parse_dates=list(date_cols))

In [2]:
print(dfs['promotions'].to_string())

   promo_id           promo_name    promo_type  promo_value start_date   end_date target_segment  city_scope                         ab_test_groups  test_allocation   success_metric
0      P000       Peak Hour Pass  surge_waiver          1.0 2025-04-26 2025-05-25            All     Nairobi                                ['All']            [1.0]  Usage Frequency
1      P001       Peak Hour Pass  surge_waiver          1.0 2025-04-26 2025-05-22            All       Cairo  ['Control', 'Variant A', 'Variant B']  [0.3, 0.4, 0.3]  Conversion Rate
2      P002       Peak Hour Pass  surge_waiver          1.0 2025-04-26 2025-05-16            All       Cairo  ['Control', 'Variant A', 'Variant B']  [0.3, 0.4, 0.3]              ROI
3      P003        Loyalty Bonus        points        100.0 2025-04-26 2025-05-04          Gold+     Nairobi  ['Control', 'Variant A', 'Variant B']  [0.3, 0.4, 0.3]  Conversion Rate
4      P004        Loyalty Bonus        points        100.0 2025-04-26 2025-05-15         

In [3]:
# Merging our dataset
trips = dfs['trips']
riders = dfs['riders']
sessions = dfs['sessions']
drivers = dfs['drivers']
promotions = dfs['promotions']

# We realise a naming mismatch in the User_id in trips and sessions
sessions.rename(columns={'rider_id':'user_id'}, inplace=True)

sessions



,session_id,user_id,session_time,time_on_app,pages_visited,converted,city,loyalty_status
0,S000000,R08605,2025-04-27 18:57:06+02:05,79,4,1,Cairo,Bronze
1,S000001,R08823,2025-04-27 07:32:22+02:27,101,3,0,Nairobi,Silver
2,S000002,R05342,2025-04-27 23:17:25+02:05,12,1,0,Cairo,Bronze
3,S000003,R05057,2025-04-27 14:40:25+00:14,19,1,0,Lagos,Silver
4,S000004,R09614,2025-04-27 08:31:22+00:14,4,1,0,Lagos,Bronze
...,...,...,...,...,...,...,...,...
49995,S049995,R03092,2025-04-27 07:52:18+00:14,76,5,0,Lagos,Bronze
49996,S049996,R02272,2025-04-27 16:52:12+02:05,44,1,1,Cairo,Bronze
49997,S049997,R06892,2025-04-27 14:02:26+02:27,9,5,0,Nairobi,Bronze
49998,S049998,R03121,2025-04-27 03:45:34+00:14,65,2,0,Lagos,Bronze


In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 20)

merged_DF = (
    trips.merge(riders, on='user_id', how='left')
    .merge(sessions, on='user_id', how='left')
    .merge(drivers, on='driver_id', how='left', suffixes=('_driver', '_rider'))
)

print('Shape:', merged_DF.shape)
merged_DF.head()


In [5]:
merged_DF

,trip_id,user_id,driver_id,fare,surge_multiplier,tip,payment_type,pickup_time,dropoff_time,pickup_lat,...,pages_visited,converted,city_driver,loyalty_status,rating,vehicle_type,signup_date_rider,last_active,city_rider,acceptance_rate
0,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
1,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
2,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
3,T000000,R05207,D00315,12.11,1.0,0.00,Card,2024-11-27 18:41:50+02:27,2024-11-27 19:33:50+02:27,-1.108123,...,1.0,0.0,Nairobi,Bronze,4.1,Sedan,2024-07-21,2025-03-21 22:47:26.558938,Nairobi,0.549628
4,T000001,R09453,D03717,8.73,1.0,0.02,Card,2024-10-28 23:13:48+00:14,2024-10-28 23:26:48+00:14,6.675266,...,5.0,0.0,Lagos,Gold,4.9,SUV,2023-05-06,2025-04-12 08:36:21.207528,Lagos,0.629250
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1001269,T199998,R02867,D00974,17.18,1.3,0.00,Mobile Money,2024-09-25 03:11:33+00:14,2024-09-25 03:45:33+00:14,6.540074,...,4.0,0.0,Lagos,Gold,3.5,Sedan,2023-09-26,2025-03-28 06:20:43.379153,Lagos,0.579188
1001270,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,...,1.0,0.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828
1001271,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,...,2.0,0.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828
1001272,T199999,R07749,D04894,13.47,1.0,0.00,Card,2024-05-24 18:19:39+02:05,2024-05-24 18:51:39+02:05,30.234277,...,1.0,1.0,Cairo,Bronze,5.0,Luxury,2023-03-30,2025-04-04 14:52:39.161225,Cairo,0.789828


To check the correlationbetween dependent and independent variables

In [6]:
# We first select the columns to deal with
numeric_cols = merged_DF.select_dtypes(include='float64').columns.tolist()
churn_correlation = merged_DF[numeric_cols].corr()['churn_prob']

churn_correlation

# We see that most almost all features has no linear relationship with the target variable 

fare                0.000277
surge_multiplier    0.000160
tip                -0.001150
pickup_lat         -0.000634
pickup_lng          0.009449
dropoff_lat        -0.000626
dropoff_lng         0.009446
age                -0.000554
avg_rating_given   -0.003537
churn_prob          1.000000
time_on_app        -0.001502
pages_visited      -0.003730
converted          -0.003654
rating             -0.002005
acceptance_rate    -0.002387
Name: churn_prob, dtype: float64

Mutual Information for Feature Selection

In [7]:
# We first need to change the churn prob into binary classification values
merged_DF['churn'] = (merged_DF['churn_prob']>0.5).astype(int)



In [8]:
merged_DF['churn'].value_counts()

churn
0    896128
1    105146
Name: count, dtype: int64

In [9]:
pd.set_option('dispaly.max_columns', None)
merged_DF

OptionError: No such keys(s): 'dispaly.max_columns'

In [39]:
merged_DF.info()

<class 'pandas.DataFrame'>
RangeIndex: 1001274 entries, 0 to 1001273
Data columns (total 37 columns):
 #   Column              Non-Null Count    Dtype         
---  ------              --------------    -----         
 0   trip_id             1001274 non-null  str           
 1   user_id             1001274 non-null  str           
 2   driver_id           1001274 non-null  str           
 3   fare                1001274 non-null  float64       
 4   surge_multiplier    1001274 non-null  float64       
 5   tip                 1001274 non-null  float64       
 6   payment_type        1001274 non-null  str           
 7   pickup_time         1001274 non-null  str           
 8   dropoff_time        1001274 non-null  str           
 9   pickup_lat          1001274 non-null  float64       
 10  pickup_lng          1001274 non-null  float64       
 11  dropoff_lat         1001274 non-null  float64       
 12  dropoff_lng         1001274 non-null  float64       
 13  weather             100

In [ ]:
from sklearn.feature_selection import mutual_info_classif
# STEPS
#1. get our features
#2. encode  catigorical features 
#3. have our x and y 
#4. fit into mutual mutual_info_classif
#5. get the mutual info scores (mi_scores)


# we drop unneeded columns
drop_cols = ['trip_id','user_id', 'driver_id','pickup_time', 'dropoff_time','signup_date_rider','session_time','session_time','session_time','churn_prob']
feature_cols = [c for c in merged_DF if c not in drop_cols]

# We select our X values
data = merged_DF[feature_cols]
# get x values and perform encoding of 
x = pd.get_dummies(data, columns=data.select_dtypes(include='object').columns.tolist(),dtype=int)

# Now we segregate x and y

x = x.drop(columns='churn')

y = data['churn']

mutual_info_score = mutual_info_classif(x,y, random_state= 42)
print(mutual_info_score)

,feature,mutual_info_score
3,pickup_lat,3.320106e-01
5,dropoff_lat,3.318310e-01
6,dropoff_lng,3.291499e-01
4,pickup_lng,3.288817e-01
7,age,3.158412e-01
13,acceptance_rate,9.028347e-02
0,fare,5.378834e-02
9,time_on_app,1.753060e-02
2,tip,1.245165e-02
8,avg_rating_given,2.111172e-03


In [ ]:
from scipy import sparse
from sklearn.feature_selection import mutual_info_classif
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

pd.set_option('display.max_rows', None)

# STEPS
# 1. get our features
# 2. encode categorical features in a memory-efficient way
# 3. build X and y from a representative sample
# 4. fit into mutual_info_classif
# 5. get the mutual info scores

# Drop unneeded columns and exclude the target column from the features.
drop_cols = [
    'trip_id', 'user_id', 'driver_id', 'pickup_time', 'dropoff_time',
    'signup_date_rider', 'session_time', 'churn_prob'
]
feature_cols = [c for c in merged_DF.columns if c not in drop_cols and c != 'churn']

data = merged_DF[feature_cols].copy()

# Convert non-numeric columns to strings so they can be encoded safely.
for col in data.columns:
    if not pd.api.types.is_numeric_dtype(data[col]):
        data[col] = data[col].astype('string').fillna('missing')

# Use a representative sample to keep the computation tractable.
sample_size = min(30000, len(data))
random_state = np.random.RandomState(42)
sample_idx = random_state.choice(len(data), size=sample_size, replace=False)

data_sample = data.iloc[sample_idx].copy()
y_sample = merged_DF['churn'].iloc[sample_idx].astype(int)

cat_cols = data_sample.select_dtypes(include='string').columns.tolist()
num_cols = data_sample.select_dtypes(exclude='string').columns.tolist()

# Impute numeric values and encode categorical values without creating a huge dense matrix.
num_imputer = SimpleImputer(strategy='median')
X_num = sparse.csr_matrix(num_imputer.fit_transform(data_sample[num_cols].astype('float32')))

if cat_cols:
    encoder = OneHotEncoder(
        handle_unknown='ignore',
        sparse_output=True,
        min_frequency=10
    )
    X_cat = encoder.fit_transform(data_sample[cat_cols])
    X = sparse.hstack([X_num, X_cat]).tocsr().astype('float32')
    encoded_feature_names = list(num_cols) + list(encoder.get_feature_names_out(cat_cols))
else:
    X = X_num.astype('float32')
    encoded_feature_names = list(num_cols)

mutual_info_score = mutual_info_classif(X, y_sample, random_state=42, n_jobs=1)

mi_df = pd.DataFrame({
    'feature': encoded_feature_names,
    'mutual_info_score': mutual_info_score
})

mi_df = mi_df.sort_values(by='mutual_info_score', ascending=False)

mi_df
